# Week 16: The Capstone — Integrating Weeks 4/5, 9, 10, and 12

Same logic as `build_capstone.py`. Nothing here is new library code — every function called already exists and is already tested: `EdgarClient` (Weeks 4–5), `chunk_document`/`get_or_create_collection`/`add_chunks`/`query_collection` (Week 9), `answer_question` (Week 10), `evaluate_retrieval`/`check_groundedness` (Week 12). This notebook's job is integration, against three real companies (AAPL, MSFT, GOOGL), in the shape `docs/projects/capstone.md` describes.

Requires `LLM_API_KEY`, `LLM_MODEL`, and `SEC_USER_AGENT` in a `.env` file (see `.env.example`). `SEC_USER_AGENT` is required by SEC EDGAR's terms of service even for the metadata-only fetch below; indexing and retrieval metrics need no API key at all — only the generation/groundedness section does.

In [ ]:
import json
import os
from functools import partial
from pathlib import Path

import httpx
from dotenv import load_dotenv

from ai_finance_course.chunking import chunk_document
from ai_finance_course.edgar import EdgarClient
from ai_finance_course.evaluation import check_groundedness, evaluate_retrieval
from ai_finance_course.rag import answer_question
from ai_finance_course.vector_store import add_chunks, get_or_create_collection, query_collection

load_dotenv()

PASSAGES_PATH = Path("data/sample/capstone_passages.json")
QUESTIONS_PATH = Path("data/sample/capstone_eval_questions.json")
PERSIST_PATH = Path("data/processed/chroma")
COLLECTION_NAME = "capstone_passages"
TICKERS = ["AAPL", "MSFT", "GOOGL"]
ANTHROPIC_MESSAGES_URL = "https://api.anthropic.com/v1/messages"


def _call_llm(prompt: str) -> str:
    with httpx.Client(timeout=60.0) as client:
        response = client.post(
            ANTHROPIC_MESSAGES_URL,
            headers={
                "x-api-key": os.environ["LLM_API_KEY"],
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
            json={
                "model": os.environ["LLM_MODEL"],
                "max_tokens": 1024,
                "messages": [{"role": "user", "content": prompt}],
            },
        )
        response.raise_for_status()
        data = response.json()
        for block in data["content"]:
            if block["type"] == "text":
                return block["text"]
        raise ValueError(f"No text block in response: {data}")

## Step 1: Real Document Discovery Against SEC EDGAR

A real HTTPS call to SEC EDGAR for three real companies — the same live-verified `EdgarClient` Weeks 4–5 already built and tested. This satisfies `capstone.md`'s "load at least three public company documents" at the discovery layer.

In [ ]:
with EdgarClient(user_agent=os.environ["SEC_USER_AGENT"]) as client:
    for ticker in TICKERS:
        filings = client.get_filings_for_ticker(ticker, limit=3)
        print(f"{ticker}:")
        for filing in filings:
            print(f"  {filing.form:8s} {filing.filing_date}  {filing.primary_document}")

## Step 2: Index the Capstone Passage Set

`data/sample/capstone_passages.json` holds short earnings/risk passages for the same three companies. This is a deliberately separate file and a deliberately separate ChromaDB collection (`capstone_passages`) from Weeks 9–15's `sample_passages` — reusing `passages.json` directly would have changed its row count and silently broken Week 15's `test_ensure_sample_index_indexes_when_empty`, which hardcodes `count() == 8`.

In [ ]:
collection = get_or_create_collection(PERSIST_PATH, COLLECTION_NAME)
passages = json.loads(PASSAGES_PATH.read_text(encoding="utf-8"))
for passage in passages:
    metadata = {"ticker": passage["ticker"], "doc_type": passage["doc_type"]}
    chunks = chunk_document(passage["text"], metadata, chunk_size=500, overlap=50)
    add_chunks(collection, chunks)

print(f"Indexed {collection.count()} chunks from {len(passages)} passages across {len(TICKERS)} companies")

## Step 3: Retrieval Metrics — No API Key Needed

In [ ]:
questions = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))

retrieval_result = evaluate_retrieval(questions, partial(query_collection, collection, n_results=3), k=3)
print(f"Questions:        {len(questions)}")
print(f"Mean recall@3:    {retrieval_result['mean_recall_at_k']:.1%}")
print(f"Mean precision@3: {retrieval_result['mean_precision_at_k']:.1%}")

retrieval_failures = [r for r in retrieval_result["per_question"] if r["recall"] < 1.0]
print(f"\nRetrieval failures ({len(retrieval_failures)}):")
for failure in retrieval_failures:
    print(f"  recall={failure['recall']:.2f} precision={failure['precision']:.2f}  {failure['query']!r}")

## Step 4: Generation and Groundedness — Needs `LLM_API_KEY`

Skips with a clear message rather than crashing partway through if no key is set — `docs/projects/rubric.md` explicitly grades "reliable retrieval, validation, error handling".

In [ ]:
if not os.environ.get("LLM_API_KEY"):
    print("LLM_API_KEY not set — skipping generation and groundedness (set it in .env to run this part).")
else:
    groundedness_failures = []
    for question in questions:
        result, evidence = answer_question(question["query"], collection, _call_llm, n_results=3)
        cited_texts = [evidence[c - 1]["text"] for c in result.citations]
        check = check_groundedness(result.answer, cited_texts, _call_llm)
        status = "OK  " if check.grounded else "FAIL"
        print(f"{status} {question['query']!r}")
        print(f"      answer: {result.answer}")
        print(f"      grounded: {check.grounded} — {check.reasoning}")
        if not check.grounded:
            groundedness_failures.append({"query": question["query"], "reasoning": check.reasoning})

    print("\n=== Summary ===")
    print(f"Retrieval failures: {len(retrieval_failures)}/{len(questions)}")
    print(f"Groundedness failures: {len(groundedness_failures)}/{len(questions)}")